In [6]:
!pip install -q transformers evaluate seqeval accelerate requests

In [7]:
import requests
import numpy as np
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

In [8]:
urls = {
    "train": "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu",
    "validation": "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu",
    "test": "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-test.conllu"
}

for split, url in urls.items():
    r = requests.get(url)
    with open(f"{split}.conllu", "w", encoding="utf-8") as f:
        f.write(r.text)

print("Files downloaded successfully!")

Files downloaded successfully!


In [10]:
def parse_conllu(filepath):
    sentences = []
    tokens = []
    tags = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if tokens:
                    sentences.append({"tokens": tokens, "tags": tags})
                    tokens = []
                    tags = []
                continue

            if line.startswith("#"):
                continue

            parts = line.split("\t")

            if "-" in parts[0] or "." in parts[0]:
                continue

            token = parts[1]
            pos_tag = parts[3]

            tokens.append(token)
            tags.append(pos_tag)

    return sentences

In [11]:
train_data = parse_conllu("train.conllu")
val_data = parse_conllu("validation.conllu")
test_data = parse_conllu("test.conllu")

print("Train samples:", len(train_data))
print("Validation samples:", len(val_data))
print("Test samples:", len(test_data))

Train samples: 12544
Validation samples: 2001
Test samples: 2077


In [12]:
dataset = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
    "test": Dataset.from_list(test_data)
})

dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 12544
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 2077
    })
})

In [13]:
all_tags = sorted(list(set(tag for example in train_data for tag in example["tags"])))
label2id = {tag: i for i, tag in enumerate(all_tags)}
id2label = {i: tag for tag, i in label2id.items()}

label_list = all_tags

print("POS Labels:", label_list)

POS Labels: ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']


In [14]:
def encode_labels(example):
    example["labels"] = [label2id[tag] for tag in example["tags"]]
    return example

dataset = dataset.map(encode_labels)
dataset["train"][0]

Map:   0%|          | 0/12544 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

{'tokens': ['Al',
  '-',
  'Zaman',
  ':',
  'American',
  'forces',
  'killed',
  'Shaikh',
  'Abdullah',
  'al',
  '-',
  'Ani',
  ',',
  'the',
  'preacher',
  'at',
  'the',
  'mosque',
  'in',
  'the',
  'town',
  'of',
  'Qaim',
  ',',
  'near',
  'the',
  'Syrian',
  'border',
  '.'],
 'tags': ['PROPN',
  'PUNCT',
  'PROPN',
  'PUNCT',
  'ADJ',
  'NOUN',
  'VERB',
  'PROPN',
  'PROPN',
  'PROPN',
  'PUNCT',
  'PROPN',
  'PUNCT',
  'DET',
  'NOUN',
  'ADP',
  'DET',
  'NOUN',
  'ADP',
  'DET',
  'NOUN',
  'ADP',
  'PROPN',
  'PUNCT',
  'ADP',
  'DET',
  'ADJ',
  'NOUN',
  'PUNCT'],
 'labels': [11,
  12,
  11,
  12,
  0,
  7,
  15,
  11,
  11,
  11,
  12,
  11,
  12,
  5,
  7,
  1,
  5,
  7,
  1,
  5,
  7,
  1,
  11,
  12,
  1,
  5,
  0,
  7,
  12]}

In [15]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [19]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [20]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
tokenized_datasets

Map:   0%|          | 0/12544 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 12544
    })
    validation: Dataset({
        features: ['tokens', 'tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['tokens', 'tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2077
    })
})

In [21]:
metric = evaluate.load("seqeval")

In [22]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [23]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [24]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [33]:
training_args = TrainingArguments(
    output_dir="./distilbert-pos",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=0.1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=1,
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [34]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
0,No log,1.318943,0.684206,0.626530,0.654099,0.695113


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=40, training_loss=1.67313232421875, metrics={'train_runtime': 508.8015, 'train_samples_per_second': 2.465, 'train_steps_per_second': 0.079, 'total_flos': 19880993470848.0, 'train_loss': 1.67313232421875, 'epoch': 0.10204081632653061})

In [36]:
results = trainer.evaluate()
results

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

{'eval_loss': 1.3189433813095093,
 'eval_precision': 0.6842056248835909,
 'eval_recall': 0.6265296550547904,
 'eval_f1': 0.6540986890427117,
 'eval_accuracy': 0.6951127371058178,
 'eval_runtime': 140.1789,
 'eval_samples_per_second': 14.275,
 'eval_steps_per_second': 0.449,
 'epoch': 0.10204081632653061}

In [37]:
sample_tokens = ["John", "is", "playing", "football", "in", "Delhi"]

inputs = tokenizer(sample_tokens, return_tensors="pt", is_split_into_words=True, truncation=True)
outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
predicted_labels = [id2label[p.item()] for p in predictions[0]]

for token, label in zip(tokens, predicted_labels):
    print(f"{token:15} -> {label}")

[CLS]           -> PRON
john            -> NOUN
is              -> AUX
playing         -> VERB
football        -> NOUN
in              -> ADP
delhi           -> NOUN
[SEP]           -> PUNCT
